In [1]:
is_interactive = True

In [2]:
import os
import ipynbname
from pathlib import Path

nb_path = ipynbname.path()
os.chdir(os.path.dirname(nb_path))
print(os.getcwd())


/home/anandas/face_recognition


In [3]:
import os
from pathlib import Path
from sqlalchemy.orm import declarative_base
from face_recognition.face_recogniser import FaceRecognizer


In [10]:
import sqlalchemy as sa
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker
from types import SimpleNamespace
def create_db(path,preserve_past:bool = True ):
    engine = create_engine(f"sqlite:///{path}", echo=True)
    SessionLocal = sessionmaker(bind=engine)
    session = SessionLocal()
    db = SimpleNamespace(
        session=session, 
        engine=engine,
        Column=sa.Column, 
        Integer=sa.Integer, 
        String=sa.String, 
        Boolean=sa.Boolean, 
        ForeignKey=sa.ForeignKey
        )  
    if not preserve_past:
        metadata = sa.MetaData() 
        inspector = sa.inspect(engine) 
        for table_name in FaceRecognizer.tables(): 
            if table_name in inspector.get_table_names(): 
                table = sa.Table(table_name, metadata, autoload_with=engine) 
                table.drop(engine)
    return db
    

In [5]:
import lancedb
def create_vector_db(path, preserve_past:bool = True ):
    db = lancedb.connect(path)
    if not preserve_past:
        for table_name in FaceRecognizer.vector_tables(): 
            if table_name in db.table_names(): 
                tbl = db.open_table(table_name)
                tbl.delete("true")
    return db

In [6]:
def setup_face_dir(face_dir: str, preserve_past: bool = True):
    os.makedirs(face_dir, exist_ok=True)
    if not preserve_past:
        for filename in os.listdir(face_dir):
            file_path = os.path.join(face_dir, filename)
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)  # remove file or symlink
    return face_dir

In [7]:
def available_faces(path: str):
    return [
        (file.stem.split("_")[0], str(file))
        for file in Path(path).rglob("*")
        if file.suffix.lower() in (".png", ".jpg", ".jpeg")
    ]

### Main module


In [8]:
rebuild_store = True
preserve_past = not rebuild_store

In [11]:
db = create_db("face_store.db", preserve_past=preserve_past)
vectordb = create_vector_db("face_vector_store.db", preserve_past=preserve_past)
Base = declarative_base()
face_dir = setup_face_dir(face_dir="face_images", preserve_past=preserve_past)

recogniser = FaceRecognizer( db=db, dbModel=Base, vectordb=vectordb, face_dir=face_dir , is_interactive = is_interactive)
Base.metadata.create_all(db.engine)

2025-08-20 14:04:07,214 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-08-20 14:04:07,215 INFO sqlalchemy.engine.Engine SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite~_%' ESCAPE '~' ORDER BY name
2025-08-20 14:04:07,216 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-08-20 14:04:07,218 INFO sqlalchemy.engine.Engine ROLLBACK
2025-08-20 14:04:07,275 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-08-20 14:04:07,276 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("face")
2025-08-20 14:04:07,277 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-08-20 14:04:07,278 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("face")
2025-08-20 14:04:07,279 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-08-20 14:04:07,280 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("person")
2025-08-20 14:04:07,280 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-08-20 14:04:07,281 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("person")
2025-08-20 14:04:07,282 

In [12]:

if rebuild_store:
    faces = available_faces(
            "/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset"
        )
    print(faces)
    recogniser.register_faces(faces)

[('Ross', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Ross_2.jpg'), ('Joey', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Joey_2.jpg'), ('Phoebe', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Phoebe_3.jpg'), ('Monica', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Monica_4.png'), ('Phoebe', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Phoebe_2.jpg'), ('Ross', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Ross_3.jpg'), ('Chandler', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Chandler_1.jpg'), ('Chandler', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Chandler_3.jpg'), ('Monica', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Monica_2.jpg'), ('Joey', '/home/anandas/demos/degirum_hailo_examples/assets/Friends_dataset/Joey_1.jpg'), ('Rachel', '/home/anandas/demos/degirum_hailo_examples/assets/Frien

InvalidRequestError: When initializing mapper Mapper[RegisteredFace(face)], expression 'Person' failed to locate a name ('Person'). If this is a class name, consider adding this relationship() to the <class 'face_recognition.store.registered_faces.db_create_table_registered_faces.<locals>.RegisteredFace'> class after both dependent classes have been defined.